As a data scientist, your company has tasked you with replicating and validating one of the AI-powered solutions explored in our book. Your mission is to rebuild the final version of the best-performing model from a given case study—whether it focuses on prediction, classification, or fraud detection—using a synthetic dataset that simulates real-world complexity. This end-to-end task includes preprocessing the data, training the final selected model (as justified in the book), saving and loading the model and its dependencies, and deploying a functional Gradio application to demonstrate your solution. You must submit your complete implementation in a single .ipynb file, ensuring reproducibility, clarity, and readiness for certification review.

# ✅ STEP 1: Install Required Libraries
Ensure you have all necessary libraries installed for the project, including scikit-learn, pandas, numpy, joblib, and gradio.

In [ ]:
!pip install --quiet scikit-learn pandas numpy joblib gradio

✅ STEP 2: Import All Required Modules
Import all libraries you will use in the project.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import gradio as gr

✅ STEP 3: Upload Dataset
Upload your dataset file (e.g., CSV) for processing.

In [ ]:
from google.colab import files
uploaded = files.upload()

# Assume the uploaded file is the only one and CSV format
import io
dataset_filename = next(iter(uploaded.keys()))
df = pd.read_csv(io.BytesIO(uploaded[dataset_filename]))
print(f"Dataset loaded with {df.shape[0]} rows and {df.shape[1]} columns.")

✅ STEP 4: Data Preprocessing
Perform cleaning, encoding categorical variables, handling missing values, feature scaling, and prepare train/test splits.

In [ ]:
# Example preprocessing steps
# 1. Handle missing values
df = df.dropna()  # or apply imputation if preferred

# 2. Identify features and target (assuming last column is target)
input_columns = df.columns[:-1].tolist()
target_column = df.columns[-1]

# 3. Encode categorical features if any
for col in input_columns:
    if df[col].dtype == 'object':
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

# 4. Encode target if necessary
if df[target_column].dtype == 'object':
    target_le = LabelEncoder()
    df[target_column] = target_le.fit_transform(df[target_column])
else:
    target_le = None

# 5. Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[input_columns])
y = df[target_column].values

# 6. Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print(f"Training samples: {X_train.shape[0]}, Testing samples: {X_test.shape[0]}")

✅ STEP 5: Train the Final Selected Model
Train the best model chosen in the book (example: Random Forest Classifier).

In [ ]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate on test set
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test set accuracy: {acc:.4f}")
print(classification_report(y_test, y_pred))

✅ STEP 6: Save Trained Model Artifacts
Save the model, scaler, and input columns for later loading.

In [ ]:
joblib.dump(model, 'final_model_case2.pkl')
joblib.dump(scaler, 'scaler_case2.pkl')
joblib.dump(input_columns, 'input_columns_case2.pkl')
if target_le:
    joblib.dump(target_le, 'target_label_encoder_case2.pkl')

✅ STEP 7: Load the Model for Prediction
Load the saved artifacts and prepare a prediction function.

In [ ]:
loaded_model = joblib.load('final_model_case2.pkl')
loaded_scaler = joblib.load('scaler_case2.pkl')
loaded_input_columns = joblib.load('input_columns_case2.pkl')
try:
    loaded_target_le = joblib.load('target_label_encoder_case2.pkl')
except FileNotFoundError:
    loaded_target_le = None

def predict(input_data):
    # Create DataFrame from input
    input_df = pd.DataFrame([input_data], columns=loaded_input_columns)
    # Apply encoding if necessary
    for col in input_df.columns:
        if input_df[col].dtype == 'object':
            # Assuming trained LabelEncoders were not saved for features, so simple encode with fit_transform
            input_df[col] = LabelEncoder().fit_transform(input_df[col])
    # Feature scaling
    input_scaled = loaded_scaler.transform(input_df)
    prediction = loaded_model.predict(input_scaled)
    if loaded_target_le:
        prediction = loaded_target_le.inverse_transform(prediction)
    return prediction[0]

✅ STEP 8: Deploy the Model Using Gradio
Create and launch an interactive web app for predictions.

In [ ]:
# Build Gradio input components based on input columns
inputs = []
for col in loaded_input_columns:
    # For simplicity, use number inputs; customize as needed
    inputs.append(gr.Number(label=col))

def gradio_predict(*args):
    input_dict = {col: val for col, val in zip(loaded_input_columns, args)}
    return predict(input_dict)

interface = gr.Interface(fn=gradio_predict,
                         inputs=inputs,
                         outputs='text',
                         title='Case Study 2: Model Prediction',
                         description='Enter feature values to get prediction from the trained model')

# Launch the Gradio app
interface.launch()

# 🔒 Submission Note:
Submit only this .ipynb notebook file.
It must include preprocessing, best model, saved files, and working Gradio app.
If preferred, you may replicate the same implementation using Python in a local Anaconda Notebook or standard Python environment. Just ensure all steps are completed and saved in a well-structured .ipynb or .py file for submission.